# Single Fidelity Surrogate Model Comparison

This workbook is aimed at testing the convergence of single fidelity models to data sets taken from analytical functions. The goal is to show how the functions increase in accuracy as the number of samples increase.

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import multi_fidelity_functions as mfs
from SurrogateModels import kriging

The following creates some scaffolding for systematically creating and assessing kriging models.

In [9]:
import time
import csv


def create_test_points(func: mfs.MultiFidelityTemplate, n=2000):
    # Create a set of samples to create a ground truth
    X_test = func.sample_random(n)
    y_test = func.high_fidelity(X_test)
    return X_test, y_test


def generate_kriging_model(
    name,
    func: mfs.MultiFidelityTemplate,
    n,
    seed,
    X_test,
    y_test,
    x_transform=True,
    bounds=None,
):
    X_train = func.sample_random(n, seed=seed)
    # X_train = func.sample_uniform(n)
    y_train = func.high_fidelity(X_train)
    # Setup timer to gauge model training time
    t0 = time.perf_counter()
    model = kriging.Kriging(
        X_train,
        y_train,
        nugget=1e-7,
        scale_input=x_transform,
        bounds=bounds,
    )
    model_training_time = time.perf_counter() - t0

    t0 = time.perf_counter()
    error = model.model_error(X_test, y_test)
    evaluation_time = time.perf_counter() - t0
    model_error = model.model_error(X_test, y_test)
    print(
        f"Run {name} :: seed {seed} :: points {n} :: error {error} :: train_time {model_training_time} complete"
    )
    return model, error, model_training_time, evaluation_time


def generate_model_comparisons(
    name,
    func: mfs.MultiFidelityTemplate,
    X_test,
    y_test,
    sample_low=5,
    sample_high=105,
    step_size=5,
    seeds=5,
):
    # Generate a Kriging model of the data and test accuracy using a number of random sample points
    n_points = list(range(sample_low, sample_high, step_size))
    points = []
    model_error = []
    training_time = []
    evaluation_time = []
    for seed in range(0, seeds):
        for n in n_points:
            X_train = func.sample_random(n, seed=seed)
            # X_train = func.sample_uniform(n)
            y_train = func.high_fidelity(X_train)
            # Setup timer to gauge model training time
            t0 = time.perf_counter()
            model = kriging.Kriging(X_train, y_train, nugget=1e-7)
            training_time.append(time.perf_counter() - t0)

            t0 = time.perf_counter()
            error = model.model_error(X_test, y_test)
            evaluation_time.append(time.perf_counter() - t0)
            model_error.append(model.model_error(X_test, y_test))
            points.append(n)
            print(
                f"Run {name} :: seed {seed} :: points {n} :: error {error} :: train_time {training_time[-1]} complete"
            )
    return points, model_error, evaluation_time, training_time


def plot_error_measure(
    points, error, name, color="blue", marker="o", label="Test", ax=None
):
    if ax == None:
        fig = plt.figure(name)
        ax = fig.add_subplot()
    ax.set_title(name)
    ax.scatter(
        points,
        error,
        color=color,
        label=label,
        marker=marker,
    )
    ax.set_xlabel("Samples")
    ax.set_ylabel("RMS Error")
    ax.legend()
    ax.grid()
    ax.set_yscale("log")
    return ax


def export_to_csv(output_file, points, model_error, training_time, evaluation_time):
    with open(output_file, "w", newline="") as file:
        writer = csv.writer(file)
        for i, val in enumerate(points):
            writer.writerow(
                [
                    val,
                    model_error[i],
                    training_time[i],
                    evaluation_time[i],
                ]
            )

### 4-Dimensional Dataset

In [10]:
key = "4D"
my_function = mfs.increasing_dimensions[key]()
X_test, y_test = create_test_points(my_function, n=3000)
points, model_error, evaluation_time, training_time = generate_model_comparisons(
    key,
    my_function,
    X_test=X_test,
    y_test=y_test,
    sample_low=50,
    sample_high=1100,
    step_size=100,
    seeds=3,
)

Run 4D :: seed 0 :: points 50 :: error 6.861148022592492 :: train_time 34.41862899999978 complete
Run 4D :: seed 0 :: points 150 :: error 1.3441760540272754 :: train_time 578.9486206000001 complete
Run 4D :: seed 0 :: points 250 :: error 0.9764954669789582 :: train_time 391.8686594000028 complete
Run 4D :: seed 0 :: points 350 :: error 0.7078764070782148 :: train_time 609.2383436999953 complete
Run 4D :: seed 0 :: points 450 :: error 0.29254505040459927 :: train_time 881.7108925000066 complete
Run 4D :: seed 0 :: points 550 :: error 0.22814347264600354 :: train_time 1885.2515197999965 complete
Run 4D :: seed 0 :: points 650 :: error 0.298910004315479 :: train_time 1652.8455506000028 complete
Run 4D :: seed 0 :: points 750 :: error 0.1603778085294 :: train_time 2552.1339404000028 complete
Run 4D :: seed 0 :: points 850 :: error 0.38659892811870467 :: train_time 1076.8999939999921 complete
Run 4D :: seed 0 :: points 950 :: error 0.17464445732752684 :: train_time 1810.056202599997 complet

KeyboardInterrupt: 

In [ ]:
report_name = "SingleFidelity4D_Cokriging_262105.csv"
export_to_csv(
    report_name,
    points=points,
    model_error=model_error,
    training_time=training_time,
    evaluation_time=evaluation_time,
)